In [1]:
import pymc_marketing
import arviz as az

print("PyMC-Marketing is installed")

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


PyMC-Marketing is installed


In [2]:
import requests
import pyreadr
import pandas as pd

# Download dataset
url = "https://github.com/facebookexperimental/Robyn/raw/main/R/data/dt_simulated_weekly.RData"

response = requests.get(url)

with open("../raw_data/dt_simulated_weekly.RData", "wb") as f:
    f.write(response.content)

# Load RData
result = pyreadr.read_r("../raw_data/dt_simulated_weekly.RData")

# Extract dataframe
df = result["dt_simulated_weekly"]

# Convert date
df["DATE"] = pd.to_datetime(df["DATE"])

df.head()

,DATE,revenue,tv_S,ooh_S,print_S,facebook_I,search_clicks_P,search_S,competitor_sales_B,facebook_S,events,newsletter
0,2015-11-23,2.754372e+06,22358.346667,0.0,12728.488889,2.430128e+07,0.000000,0.000000,8125009,7607.132915,na,19401.653846
1,2015-11-30,2.584277e+06,28613.453333,0.0,0.000000,5.527033e+06,9837.238486,4133.333333,7901549,1141.952450,na,14791.000000
2,2015-12-07,2.547387e+06,0.000000,132278.4,453.866667,1.665159e+07,12044.119653,3786.666667,8300197,4256.375378,na,14544.000000
3,2015-12-14,2.875220e+06,83450.306667,0.0,17680.000000,1.054977e+07,12268.070319,4253.333333,8122883,2800.490677,na,2800.000000
4,2015-12-21,2.215953e+06,0.000000,277336.0,0.000000,2.934090e+06,9467.248023,3613.333333,7105985,689.582605,na,15478.000000


In [3]:
channel_columns = [
    "tv_S",
    "ooh_S",
    "print_S",
    "facebook_S",
    "search_S"
]

control_columns = [
    "competitor_sales_B",
    "newsletter"
]

target_column = "revenue"
date_column = "DATE"

df_model = df[[date_column, target_column] + channel_columns + control_columns].copy()

df_model.head()

,DATE,revenue,tv_S,ooh_S,print_S,facebook_S,search_S,competitor_sales_B,newsletter
0,2015-11-23,2.754372e+06,22358.346667,0.0,12728.488889,7607.132915,0.000000,8125009,19401.653846
1,2015-11-30,2.584277e+06,28613.453333,0.0,0.000000,1141.952450,4133.333333,7901549,14791.000000
2,2015-12-07,2.547387e+06,0.000000,132278.4,453.866667,4256.375378,3786.666667,8300197,14544.000000
3,2015-12-14,2.875220e+06,83450.306667,0.0,17680.000000,2800.490677,4253.333333,8122883,2800.000000
4,2015-12-21,2.215953e+06,0.000000,277336.0,0.000000,689.582605,3613.333333,7105985,15478.000000


In [4]:
df_model.info()

<class 'pandas.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype        
---  ------              --------------  -----        
 0   DATE                208 non-null    datetime64[s]
 1   revenue             208 non-null    float64      
 2   tv_S                208 non-null    float64      
 3   ooh_S               208 non-null    float64      
 4   print_S             208 non-null    float64      
 5   facebook_S          208 non-null    float64      
 6   search_S            208 non-null    float64      
 7   competitor_sales_B  208 non-null    int32        
 8   newsletter          208 non-null    float64      
dtypes: datetime64[s](1), float64(7), int32(1)
memory usage: 13.9 KB


In [5]:
from pymc_marketing.mmm import (
    MMM,
    GeometricAdstock,
    LogisticSaturation
)

In [8]:
from sklearn.preprocessing import MaxAbsScaler

df_model = df_model.sort_values("DATE").reset_index(drop=True)

train_size = int(len(df_model) * 0.8)

train_df = df_model.iloc[:train_size].copy()
test_df = df_model.iloc[train_size:].copy()

channel_scaler = MaxAbsScaler()
control_scaler = MaxAbsScaler()
target_scaler = MaxAbsScaler()

train_df[channel_columns] = channel_scaler.fit_transform(train_df[channel_columns])
test_df[channel_columns] = channel_scaler.transform(test_df[channel_columns])

train_df[control_columns] = control_scaler.fit_transform(train_df[control_columns])
test_df[control_columns] = control_scaler.transform(test_df[control_columns])

train_df["revenue_scaled"] = target_scaler.fit_transform(train_df[["revenue"]])
test_df["revenue_scaled"] = target_scaler.transform(test_df[["revenue"]])

train_df.head()

,DATE,revenue,tv_S,ooh_S,print_S,facebook_S,search_S,competitor_sales_B,newsletter,revenue_scaled
0,2015-11-23,2.754372e+06,0.141467,0.000000,0.398733,0.547251,0.000000,0.813743,0.201605,0.719623
1,2015-11-30,2.584277e+06,0.181044,0.000000,0.000000,0.082151,0.241058,0.791362,0.153695,0.675183
2,2015-12-07,2.547387e+06,0.000000,0.264366,0.014218,0.306200,0.220840,0.831288,0.151128,0.665545
3,2015-12-14,2.875220e+06,0.528011,0.000000,0.553845,0.201465,0.248056,0.813530,0.029095,0.751197
4,2015-12-21,2.215953e+06,0.000000,0.554271,0.000000,0.049608,0.210731,0.711684,0.160834,0.578953


In [9]:
mmm = MMM(
    date_column="DATE",
    channel_columns=channel_columns,
    control_columns=control_columns,
    adstock=GeometricAdstock(l_max=8),
    saturation=LogisticSaturation()
)

c:\Users\ial\Desktop\seminar-mmm-pfn\.venv\Lib\site-packages\pydantic\_internal\_validate_call.py:137: FutureWarning: 
            The MMM class is deprecated and will be removed in a future version (in version 0.20.0).
            Please use the multidimensional MMM class instead.
            That is, `from pymc_marketing.mmm.multidimensional import MMM`.
            All our documentation has been updated to reflect this change.
            Refer to the migration guide for more details: https://www.pymc-marketing.io/en/latest/notebooks/mmm/mmm_migration_guide.html
            
  res = self.__pydantic_validator__.validate_python(pydantic_core.ArgsKwargs(args, kwargs))


In [ ]:
mmm.fit(
    X=train_df,
    y=train_df["revenue_scaled"],
    draws=500,
    tune=500,
    chains=2,
    cores=1,
    target_accept=0.9,
    random_seed=42
)

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [intercept, adstock_alpha, saturation_lam, saturation_beta, gamma_control, y_sigma]


Output()